In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold

In [2]:
# Cargar archivo de entrenamiento
df_train = pd.read_csv('hoteles-entrena.csv')

In [3]:
# Convertir la columna 'arrival_date' a formato datetime
df_train['arrival_date'] = pd.to_datetime(df_train['arrival_date'], format='%Y-%m-%d')

# Extraer año, mes y día
df_train['arrival_year'] = df_train['arrival_date'].dt.year
df_train['arrival_month'] = df_train['arrival_date'].dt.month
df_train['arrival_day'] = df_train['arrival_date'].dt.day

# Crear la columna day_of_year
df_train['day_of_year'] = df_train['arrival_date'].dt.dayofyear

# Añadir componentes seno y coseno para capturar estacionalidad
df_train['arrival_month_sin'] = np.sin(2 * np.pi * df_train['arrival_month'] / 12)
df_train['arrival_month_cos'] = np.cos(2 * np.pi * df_train['arrival_month'] / 12)

# Añadir componentes seno y coseno para capturar estacionalidad en day_of_year
df_train['day_of_year_sin'] = np.sin(2 * np.pi * df_train['day_of_year'] / 365)
df_train['day_of_year_cos'] = np.cos(2 * np.pi * df_train['day_of_year'] / 365)

# Visualizar las primeras filas
df_train[['arrival_date', 'arrival_year', 'arrival_month', 'arrival_day', 
          'arrival_month_sin', 'arrival_month_cos', 'day_of_year', 
          'day_of_year_sin', 'day_of_year_cos']].head()

# Sumar stays_in_weekend_nights y stays_in_week_nights como total de noches
df_train['total_nights'] = df_train['stays_in_weekend_nights'] + df_train['stays_in_week_nights']

# Crear la columna stays_in_weekend (1 si tiene noches de fin de semana, 0 en caso contrario)
df_train['stays_in_weekend'] = (df_train['stays_in_weekend_nights'] > 0).astype(int)

# Crear la columna 'stay_days' como tipo 'object' y asignar un valor inicial
df_train['stay_days'] = pd.Series('both', index=df_train.index, dtype='object')

# Asignar valores condicionales
df_train.loc[(df_train['stays_in_weekend_nights'] > 0) & (df_train['stays_in_week_nights'] == 0), 'stay_days'] = 'weekend'
df_train.loc[(df_train['stays_in_weekend_nights'] == 0) & (df_train['stays_in_week_nights'] > 0), 'stay_days'] = 'weekdays'

# Añadir una columna weekday para el día de la semana de la llegada numérico
df_train['weekday'] = df_train['arrival_date'].dt.weekday

# Reemplazar valores NA con valores específicos para cada columna
df_train['country'] = df_train['country'].fillna('NON')
df_train['agent'] = df_train['agent'].fillna(0)
df_train['company'] = df_train['company'].fillna(0)

# Normalizar

# Convertir 'children' a binario
df_train['children'] = df_train['children'].map({'children': 1, 'none': 0})

# Normalizar columnas con Min-Max Scaling
scaler = MinMaxScaler()
cols_min_max = ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 
                'adults', 'previous_cancellations', 'previous_bookings_not_canceled', 
                'booking_changes', 'days_in_waiting_list', 'average_daily_rate', 'total_nights']
df_train[cols_min_max] = scaler.fit_transform(df_train[cols_min_max])

# Mapear los valores en 'hotel' y 'required_car_parking_spaces' a 1 y 0 sin advertencia
df_train['hotel'] = df_train['hotel'].map({'Resort_Hotel': 1, 'City_Hotel': 0}).astype(int)
df_train['required_car_parking_spaces'] = df_train['required_car_parking_spaces'].map({'parking': 1, 'none': 0}).astype(int)

# Eliminar 'arrival_date'
df_train.drop(columns=['arrival_date'], inplace=True)

# Definir función de Target Encoding con K-Fold y manejo de NaN usando promedio global
def target_encoding_kfold_inplace(data, column, target, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    global_mean = data[target].mean()
    
    for train_index, val_index in kf.split(data):
        train_fold, val_fold = data.iloc[train_index], data.iloc[val_index]
        mean_target = train_fold.groupby(column)[target].mean()
        
        # Asignar valores mapeados y rellenar NaN con el promedio global
        data.loc[val_index, column] = val_fold[column].map(mean_target).fillna(global_mean)

# Aplicar Target Encoding in-place a cada columna
cols_target_encoding = ['meal', 'country', 'market_segment', 'distribution_channel', 
                        'reserved_room_type', 'assigned_room_type', 'deposit_type', 
                        'agent', 'company', 'customer_type', 'stay_days']

for col in cols_target_encoding:
    target_encoding_kfold_inplace(df_train, col, 'children')

# Exportar a un archivo CSV
df_train.to_csv('hoteles-entrena-limpio-normalizado.csv', index=False)